### train_dl.py
#### DL baseline
- Load shared dataset + shared split
- Tokenizer
- Pad sequence
- Label encoding
- Class weights
- Train BiLSTM
- Save DL model

In [1]:
import os
import json
import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

### Files

In [2]:
SHARED_DATASET_FILE = "../data/processed/cleaned_emotions_shared.csv"
SHARED_SPLIT_FILE = "../data/splits/shared_split.json"

DL_MODEL_FILE = "../models/bilstm_model.keras"
WORD_INDEX_FILE = "../models/word_index.json"
LABEL_MAPPING_FILE = "../models/label_mapping.json"
DL_METADATA_FILE = "../models/dl_metadata.json"
DL_PRED_FILE = "../data/processed/dl_test_predictions.csv"

MAX_VOCAB_SIZE = 10000

os.makedirs("../models", exist_ok=True)
os.makedirs("../data/processed", exist_ok=True)

### Load shared data

In [3]:
df = pd.read_csv(SHARED_DATASET_FILE)

with open(SHARED_SPLIT_FILE, "r", encoding="utf-8") as f:
    split_dict = json.load(f)

train_df = df[df["row_id"].isin(split_dict["train_ids"])].copy()
val_df = df[df["row_id"].isin(split_dict["val_ids"])].copy()
test_df = df[df["row_id"].isin(split_dict["test_ids"])].copy()

print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)


Train: (1512, 3)
Val  : (324, 3)
Test : (324, 3)


### Tokenizer

In [4]:
tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(train_df["text"])

word_index = tokenizer.word_index

X_train_seq = tokenizer.texts_to_sequences(train_df["text"])
X_val_seq = tokenizer.texts_to_sequences(val_df["text"])
X_test_seq = tokenizer.texts_to_sequences(test_df["text"])

sequence_lengths = [len(seq) for seq in X_train_seq]
max_sequence_length = int(np.percentile(sequence_lengths, 95))

print("\nVocabulary size:", len(word_index))
print("Max sequence length:", max_sequence_length)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_sequence_length, padding="post", truncating="post")
X_val_pad = pad_sequences(X_val_seq, maxlen=max_sequence_length, padding="post", truncating="post")
X_test_pad = pad_sequences(X_test_seq, maxlen=max_sequence_length, padding="post", truncating="post")

print("X_train_pad:", X_train_pad.shape)
print("X_val_pad  :", X_val_pad.shape)
print("X_test_pad :", X_test_pad.shape)



Vocabulary size: 1349
Max sequence length: 38
X_train_pad: (1512, 38)
X_val_pad  : (324, 38)
X_test_pad : (324, 38)


### Encode labels

In [5]:
label_encoder = LabelEncoder()

y_train = label_encoder.fit_transform(train_df["label"])
y_val = label_encoder.transform(val_df["label"])
y_test = label_encoder.transform(test_df["label"])

num_classes = len(label_encoder.classes_)

y_train_cat = to_categorical(y_train, num_classes=num_classes)
y_val_cat = to_categorical(y_val, num_classes=num_classes)
y_test_cat = to_categorical(y_test, num_classes=num_classes)

print("\nClasses:", label_encoder.classes_)


Classes: ['angry' 'disgust' 'fear' 'happy' 'neutral' 'sad']


### Class weights

In [6]:
class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)
class_weights = dict(enumerate(class_weights_array))

print("Class weights:", class_weights)

Class weights: {0: np.float64(1.105263157894737), 1: np.float64(0.842809364548495), 2: np.float64(0.9130434782608695), 3: np.float64(1.5365853658536586), 4: np.float64(0.72), 5: np.float64(1.2923076923076924)}


### Build model

In [7]:
vocab_size = min(MAX_VOCAB_SIZE, len(word_index) + 1)

model = Sequential()
model.add(Embedding(input_dim=vocab_size, output_dim=128, input_length=max_sequence_length))
model.add(Bidirectional(LSTM(64)))
model.add(Dropout(0.5))
model.add(Dense(64, activation="relu"))
model.add(Dropout(0.3))
model.add(Dense(num_classes, activation="softmax"))

model.compile(
    loss="categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

model.summary()

c:\Users\DELL\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

### Train model

In [8]:
early_stop = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6)

history = model.fit(
    X_train_pad,
    y_train_cat,
    validation_data=(X_val_pad, y_val_cat),
    epochs=20,
    batch_size=16,
    callbacks=[early_stop, reduce_lr],
    class_weight=class_weights,
    verbose=1
)

Epoch 1/20
95/95 ━━━━━━━━━━━━━━━━━━━━ 7s 29ms/step - accuracy: 0.3981 - loss: 1.6203 - val_accuracy: 0.6420 - val_loss: 1.0794 - learning_rate: 0.0010
Epoch 2/20
95/95 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.7328 - loss: 0.7536 - val_accuracy: 0.8519 - val_loss: 0.4198 - learning_rate: 0.0010
Epoch 3/20
95/95 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - accuracy: 0.8836 - loss: 0.2972 - val_accuracy: 0.8519 - val_loss: 0.3313 - learning_rate: 0.0010
Epoch 4/20
95/95 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.9107 - loss: 0.2113 - val_accuracy: 0.8796 - val_loss: 0.2690 - learning_rate: 0.0010
Epoch 5/20
95/95 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.9425 - loss: 0.1416 - val_accuracy: 0.8889 - val_loss: 0.2547 - learning_rate: 0.0010
Epoch 6/20
95/95 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.9597 - loss: 0.1017 - val_accuracy: 0.9136 - val_loss: 0.2676 - learning_rate: 0.0010
Epoch 7/20
95/95 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.9696 - loss: 0.0687 - val_acc

### Evaluate

In [9]:
test_loss, test_acc = model.evaluate(X_test_pad, y_test_cat, verbose=0)

y_pred_prob = model.predict(X_test_pad, verbose=0)
y_pred_classes = np.argmax(y_pred_prob, axis=1)

y_pred_labels = label_encoder.inverse_transform(y_pred_classes)
y_true_labels = label_encoder.inverse_transform(y_test)

acc = accuracy_score(y_true_labels, y_pred_labels)
f1_macro = f1_score(y_true_labels, y_pred_labels, average="macro")
f1_weighted = f1_score(y_true_labels, y_pred_labels, average="weighted")

print("\n===== DL BASELINE (BiLSTM) =====")
print(f"Test Loss     : {test_loss:.4f}")
print(f"Accuracy      : {acc:.4f}")
print(f"F1 Macro      : {f1_macro:.4f}")
print(f"F1 Weighted   : {f1_weighted:.4f}")

print("\nClassification Report:")
print(classification_report(y_true_labels, y_pred_labels))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true_labels, y_pred_labels, labels=label_encoder.classes_))


===== DL BASELINE (BiLSTM) =====
Test Loss     : 0.3309
Accuracy      : 0.9012
F1 Macro      : 0.9129
F1 Weighted   : 0.8993

Classification Report:
              precision    recall  f1-score   support

       angry       0.92      1.00      0.96        49
     disgust       0.90      0.95      0.92        64
        fear       0.86      0.72      0.78        60
       happy       1.00      0.97      0.99        35
     neutral       0.83      0.87      0.85        75
         sad       0.98      0.98      0.98        41

    accuracy                           0.90       324
   macro avg       0.92      0.91      0.91       324
weighted avg       0.90      0.90      0.90       324


Confusion Matrix:
[[49  0  0  0  0  0]
 [ 1 61  0  0  2  0]
 [ 1  5 43  0 11  0]
 [ 0  0  1 34  0  0]
 [ 2  1  6  0 65  1]
 [ 0  1  0  0  0 40]]


### Save model files

In [10]:
model.save(DL_MODEL_FILE)

with open(WORD_INDEX_FILE, "w", encoding="utf-8") as f:
    json.dump(tokenizer.word_index, f, ensure_ascii=False, indent=2)

label_mapping = {label: int(i) for i, label in enumerate(label_encoder.classes_)}
with open(LABEL_MAPPING_FILE, "w", encoding="utf-8") as f:
    json.dump(label_mapping, f, ensure_ascii=False, indent=2)

metadata = {
    "model_name": "BiLSTM",
    "max_vocab_size": MAX_VOCAB_SIZE,
    "max_sequence_length": int(max_sequence_length),
    "num_classes": int(num_classes),
    "classes": label_encoder.classes_.tolist()
}

with open(DL_METADATA_FILE, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("\nSaved DL model files")


Saved DL model files


### Save DL predictions

In [11]:
pred_df = test_df[["row_id", "text", "label"]].copy()
pred_df["dl_pred"] = y_pred_labels
pred_df["dl_confidence"] = y_pred_prob.max(axis=1)

proba_cols = pd.DataFrame(y_pred_prob, columns=[f"dl_proba_{c}" for c in label_encoder.classes_])
pred_df = pd.concat([pred_df.reset_index(drop=True), proba_cols.reset_index(drop=True)], axis=1)

pred_df.to_csv(DL_PRED_FILE, index=False, encoding="utf-8-sig")
print("Saved DL predictions:", DL_PRED_FILE)

Saved DL predictions: ../data/processed/dl_test_predictions.csv
